# export_weights
- https://github.com/ISEEKYAN/mbridge/blob/main/mbridge/core/bridge.py#L905
- 它的主要任务是：<font color='red'>在分布式训练环境下，将 Megatron（Mcore）模型内部被切分、分散的权重，高效地收集、合并并转换
  为推理引擎（如 SGLang）能够识别的标准格式。</font>
- export_weights 本质上是一个高度优化的分布式权重“搬运与重组”引擎。它不仅处理了从 Megatron 到标准格式的命名映射，更通过 bucketed_all_gather 等底层通信原语，高效解决了大模型在 TP/EP/PP 等多维并行切分下，权重分散存储的难题，最终为推理引擎提供了一份完整、标准且连续的模型权重。

keep_stacked_experts：这是一个针对 MoE（混合专家）模型 的特有参数。
- 如果为 True（默认），导出的 MoE 专家权重会保持 Hugging Face 格式的“堆叠”状态（即所有专家的权重合并成一个大张量）。
- 如果为 False，则会将每个专家拆分成独立的张量导出（但门控和上投影矩阵 gate_up_proj 依然保持融合状态）。

In [ ]:
from .util import (
    bucketed_all_gather_into_tensor,
    bucketed_pp_broadcast,
    get_model,
    iter_model_named_params,
    unwrap_model,
) 

    def export_weights(
        self,
        models: list[torch.nn.Module], # 传入的是 Megatron 训练框架中的模型对象列表
        keep_stacked_experts: bool = True,
    ) -> Generator[tuple[str, torch.Tensor], None, None]:
        """
        Export weights from models.

        Args:
            models: List of models to export weights from
            keep_stacked_experts: If True (default), keep hf format of stacked experts; else emit each expert as its own hf key, yet keep gate_up_proj fused

        Returns:
            Generator[tuple[str, torch.Tensor]]: Generator of (name, tensor) tuples.
            返回一个 Python 生成器（Generator），每次迭代产出一个 (权重名称, 权重张量) 的元组。
            使用生成器可以避免一次性将所有权重加载到内存中，极大地节省了显存和内存开销。
        """
        
        '''
        断言检查：确保内部用于暂存权重的缓冲区 export_weights_buff 是空的，防止上一次导出的残留数据干扰本次操作。
        '''
        assert (
            len(self.export_weights_buff) == 0
        ), f"should be empty {self.export_weights_buff=}"
        
        '''
        解包模型：unwrap_model 的作用是剥离掉分布式训练时包裹在模型外层的壳（如 DistributedDataParallel 或 Megatron 的特定包装器），
        从而获取到最底层的、包含真实参数的 torch.nn.Module
        '''
        models = [unwrap_model(model) for model in models]
        '''
        判断当前程序是否运行在多卡/多机的分布式环境下
        '''
        is_distributed = (
            torch.distributed.is_available() and torch.distributed.is_initialized()
        )

        '''
        定义并行组与数据收集器
        '''
        groups_by_mode = {
            "tp": (self.mpu.tp_group, self.mpu.tp_size), # 将大矩阵切分到多张卡上计算
            "ep": (self.mpu.ep_group, self.mpu.ep_size), # 专家并行，MoE 模型中用于将不同的专家分配到不同卡上
            "etp": (self.mpu.etp_group, self.mpu.etp_size), # 专家张量并行，MoE 专家内部的张量切分
        }

        '''
        _collect_bucket 内部函数：这是核心的“收集器”。它接收一批（bucket）被切分的权重，根据指定的并行模式（如 tp），
        调用 bucketed_all_gather_into_tensor 这个高效的通信原语，将分散在各个 GPU 上的权重碎片跨卡收集并拼接成完整的权重张量。
        '''
        def _collect_bucket(bucket: list[tuple[str, torch.Tensor]], parallel_mode: str):
            if parallel_mode not in groups_by_mode:
                raise ValueError(f"Unsupported parallel_mode: {parallel_mode}")
            group, group_size = groups_by_mode[parallel_mode]
            return bucketed_all_gather_into_tensor(
                bucket,
                group=group,
                group_size=group_size,
                per_rank_bucket_size_bytes=self._get_collective_bucket_size_bytes(
                    group_size
                ),
            )

        '''
        核心导出流程；
        
        跨流水线并行组广播 (pp group)：_iter_all_ranks_named_params 负责处理流水线并行（Pipeline Parallelism）。
        在 PP 模式下，模型的不同层分布在不同机器上，这一步确保能遍历到所有 ranks 上的参数，并在 PP 组内进行必要的广播同步。
        '''
        # broadcast inside pp group
        named_params = self._iter_all_ranks_named_params(models, is_distributed)
        
        '''
        - 跨张量/专家并行组收集与产出：_iter_bucketed_export_outputs 是最终的导出执行者。
            - 它会遍历上一步获取的所有参数。
            - 将需要合并的参数打包成“桶”（bucket），交给前面定义的 _collect_bucket 函数进行跨卡收集（AllGather）。
            - 最后，将收集好的完整权重，按照推理引擎要求的标准命名规则，通过 yield 逐个产出。
        '''
        # allgather inside tp/ep/etp groups
        yield from self._iter_bucketed_export_outputs(
            named_params,
            _collect_bucket,
            keep_stacked_experts=keep_stacked_experts,
        )
